<a href="https://colab.research.google.com/github/Latryna/titans-legal-docs/blob/main/Multi_LLM_Chat_Python_FastAPI_(z_obs%C5%82ug%C4%85_audio).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# main.py
import os
import asyncio
import json
import logging
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from fastapi.responses import HTMLResponse
from fastapi.middleware.cors import CORSMiddleware
from azure.core.exceptions import AzureError
from azure.ai.openai.aio import OpenAIClient
from azure.ai.openai import AzureKeyCredential
import httpx
from dotenv import load_dotenv

# --- Konfiguracja i Inicjalizacja ---
load_dotenv()
logging.basicConfig(level=logging.INFO)

app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # W produkcji zawęzić do konkretnej domeny
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# --- Klienci API ---
try:
    azure_client = OpenAIClient(
        endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        credential=AzureKeyCredential(os.getenv("AZURE_OPENAI_KEY")),
        api_version="2024-02-01"
    )
    AZURE_DEPLOYMENT = os.getenv("AZURE_DEPLOYMENT_NAME", "deployment-copilot")
except Exception as e:
    logging.error(f"Failed to initialize Azure Client: {e}")
    azure_client = None

GEMINI_API_URL = "https://generativelanguage.googleapis.com/v1beta/models/gemini-pro:streamGenerateContent"
GEMINI_API_KEY = os.getenv("GOOGLE_API_KEY")

# --- Logika Strumieniowania Modeli ---

async def stream_azure(history: list, send_func) -> str:
    """Strumieniuje odpowiedź z Azure OpenAI i zwraca pełną treść."""
    if not azure_client:
        error_msg = "Klient Azure OpenAI nie jest skonfigurowany."
        await send_func(json.dumps({"model": "azure", "error": error_msg}))
        return f"[ERROR: {error_msg}]"

    full_response = ""
    try:
        stream = await azure_client.get_chat_completions(
            model=AZURE_DEPLOYMENT,
            messages=history,
            stream=True
        )
        async for event in stream:
            if event.choices:
                delta = event.choices[0].delta.content
                if delta:
                    full_response += delta
                    await send_func(json.dumps({"model": "azure", "text": delta}))
    except AzureError as e:
        logging.error(f"Azure API Error: {e}")
        error_msg = f"Błąd API Azure: {e.message or e.reason}"
        await send_func(json.dumps({"model": "azure", "error": error_msg}))
        return f"[ERROR: {error_msg}]"
    except Exception as e:
        logging.error(f"Unexpected error in stream_azure: {e}")
        error_msg = "Wystąpił nieoczekiwany błąd po stronie Azure."
        await send_func(json.dumps({"model": "azure", "error": error_msg}))
        return f"[ERROR: {error_msg}]"
    finally:
        await send_func(json.dumps({"model": "azure", "done": True}))
    return full_response

def convert_history_to_gemini(history: list) -> list:
    """Konwertuje format historii z OpenAI na format Gemini."""
    gemini_history = []
    for msg in history:
        role = "model" if msg["role"] == "assistant" else "user"
        gemini_history.append({"role": role, "parts": [{"text": msg["content"]}]})
    return gemini_history

async def stream_gemini(history: list, send_func) -> str:
    """Strumieniuje odpowiedź z Google Gemini i zwraca pełną treść."""
    if not GEMINI_API_KEY:
        error_msg = "Klucz API Google Gemini nie jest skonfigurowany."
        await send_func(json.dumps({"model": "gemini", "error": error_msg}))
        return f"[ERROR: {error_msg}]"

    full_response = ""
    gemini_contents = convert_history_to_gemini(history)

    async with httpx.AsyncClient(timeout=120.0) as client:
        try:
            async with client.stream(
                "POST",
                f"{GEMINI_API_URL}?key={GEMINI_API_KEY}",
                json={"contents": gemini_contents},
                headers={"Content-Type": "application/json"},
            ) as response:
                response.raise_for_status()
                buffer = ""
                async for chunk in response.aiter_bytes():
                    buffer += chunk.decode("utf-8")
                    while '}\n' in buffer:
                        json_str, buffer = buffer.split('}\n', 1)
                        json_str += '}'
                        try:
                            if json_str.strip().startswith("data:"):
                                json_str = json_str.strip()[5:]
                            part = json.loads(json_str)
                            if (text := part.get("candidates", [{}])[0].get("content", {}).get("parts", [{}])[0].get("text")):
                                full_response += text
                                await send_func(json.dumps({"model": "gemini", "text": text}))
                        except (json.JSONDecodeError, IndexError, KeyError) as e:
                            logging.warning(f"Could not parse Gemini chunk: {e} | Chunk: '{json_str}'")
                            continue
        except httpx.HTTPStatusError as e:
            logging.error(f"Gemini API Error {e.response.status_code}: {e.response.text}")
            error_msg = f"Błąd API Gemini (status: {e.response.status_code})"
            await send_func(json.dumps({"model": "gemini", "error": error_msg}))
            return f"[ERROR: {error_msg}]"
        except Exception as e:
            logging.error(f"Unexpected error in stream_gemini: {e}")
            error_msg = "Wystąpił nieoczekiwany błąd po stronie Gemini."
            await send_func(json.dumps({"model": "gemini", "error": error_msg}))
            return f"[ERROR: {error_msg}]"
        finally:
            await send_func(json.dumps({"model": "gemini", "done": True}))
    return full_response

# --- Endpointy FastAPI ---

@app.get("/", response_class=HTMLResponse)
async def get():
    with open("index.html", "r", encoding="utf-8") as f:
        return HTMLResponse(content=f.read())

@app.websocket("/chat")
async def websocket_endpoint(websocket: WebSocket):
    await websocket.accept()
    history = []
    audio_buffer = bytearray()

    try:
        while True:
            message = await websocket.receive()

            if "text" in message:
                data = json.loads(message["text"])
                if data.get("type") == "text":
                    user_message = data.get("payload", "")
                    logging.info(f"Received text: {user_message}")
                    history.append({"role": "user", "content": user_message})

                    async def send_to_client(msg):
                        await websocket.send_text(msg)

                    azure_task = asyncio.create_task(stream_azure(history, send_to_client))
                    gemini_task = asyncio.create_task(stream_gemini(history, send_to_client))

                    azure_response, gemini_response = await asyncio.gather(azure_task, gemini_task)

                    if azure_response and not azure_response.startswith("[ERROR"):
                         history.append({"role": "assistant", "content": azure_response})
                    elif gemini_response and not gemini_response.startswith("[ERROR"):
                        history.append({"role": "assistant", "content": gemini_response})

            elif "bytes" in message:
                audio_data = message["bytes"]
                logging.info(f"Received {len(audio_data)} bytes of audio data.")
                audio_buffer.extend(audio_data)
                # KROK 2: W tym miejscu dane audio (audio_buffer) byłyby przekazywane
                # do modelu Speech-to-Text (np. Whisper) w celu transkrypcji.
                # text_from_audio = await speech_to_text_model(audio_buffer)
                # a następnie text_from_audio byłby dodany do 'history' i wysłany do LLM.

    except WebSocketDisconnect:
        logging.info("Client disconnected.")
        # Po rozłączeniu zapisz zbuforowane audio do pliku
        if audio_buffer:
            with open("received_audio.webm", "wb") as f:
                f.write(audio_buffer)
            logging.info("Saved buffered audio to received_audio.webm")
    except Exception as e:
        logging.error(f"An error occurred in the websocket connection: {e}", exc_info=True)
        await websocket.close(code=1011, reason="An internal server error occurred.")

# Uruchomienie serwera: uvicorn main:app --reload --port 8000
```html
<!-- index.html -->
<!DOCTYPE html>
<html lang="pl">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Multi-LLM Chat</title>
  <style>
    :root {
      --background-color: #1a1a1a;
      --text-color: #e0e0e0;
      --border-color: #333;
      --input-bg: #252525;
      --user-msg-bg: #004a7c;
      --azure-msg-bg: #0b5345;
      --gemini-msg-bg: #512e5f;
      --error-color: #e74c3c;
      --record-btn-bg: #c0392b;
      --record-btn-active-bg: #e74c3c;
    }
    body {
      font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif;
      margin: 0;
      background: var(--background-color);
      color: var(--text-color);
      display: flex;
      flex-direction: column;
      height: 100vh;
    }
    .chat-wrapper {
      flex: 1;
      display: flex;
      flex-direction: column;
      max-width: 800px;
      width: 100%;
      margin: 0 auto;
      padding: 1rem;
      box-sizing: border-box;
    }
    #chat-container {
      flex-grow: 1;
      overflow-y: auto;
      padding: 0 1rem;
      border: 1px solid var(--border-color);
      border-radius: 8px;
      margin-bottom: 1rem;
    }
    .message {
      margin: 1rem 0;
      padding: 0.8em 1.2em;
      border-radius: 12px;
      line-height: 1.6;
      max-width: 80%;
      word-wrap: break-word;
    }
    .user { background: var(--user-msg-bg); align-self: flex-end; margin-left: auto; }
    .model-response { align-self: flex-start; margin-right: auto; }
    .azure { background: var(--azure-msg-bg); }
    .gemini { background: var(--gemini-msg-bg); }
    .model-label { font-weight: bold; font-size: 0.8em; opacity: 0.8; display: block; margin-bottom: 0.3em; text-transform: uppercase; }
    .error { color: var(--error-color); font-weight: bold; }
    .typing::after { content: '▋'; animation: blink 1s step-end infinite; display: inline-block; margin-left: 0.2em; }
    @keyframes blink { 50% { opacity: 0; } }
    .input-controls { display: flex; align-items: center; }
    #input-area { display: flex; flex-grow: 1; }
    #input { flex-grow: 1; padding: 0.8em; border: 1px solid var(--border-color); background: var(--input-bg); color: var(--text-color); border-radius: 8px; resize: none; font-size: 1em; }
    #send { padding: 0.8em 1.5em; margin-left: 1em; border: none; background: var(--user-msg-bg); color: white; cursor: pointer; border-radius: 8px; font-size: 1em; }
    #record { width: 48px; height: 48px; margin-right: 1em; border: none; background: var(--record-btn-bg); color: white; cursor: pointer; border-radius: 50%; font-size: 1.5em; display: flex; align-items: center; justify-content: center; transition: background-color 0.2s; }
    #record.recording { background-color: var(--record-btn-active-bg); animation: pulse 1.5s infinite; }
    #status { margin-top: 0.5em; font-size: 0.9em; color: #aaa; height: 1em; }
    @keyframes pulse { 0% { box-shadow: 0 0 0 0 rgba(231, 76, 60, 0.7); } 70% { box-shadow: 0 0 0 10px rgba(231, 76, 60, 0); } 100% { box-shadow: 0 0 0 0 rgba(231, 76, 60, 0); } }
  </style>
</head>
<body>
  <div class="chat-wrapper">
    <h1>Multi-LLM Chat</h1>
    <div id="status">Gotowy</div>
    <div id="chat-container"></div>
    <div class="input-controls">
      <button id="record">●</button>
      <div id="input-area">
        <textarea id="input" rows="1" placeholder="Wpisz wiadomość..."></textarea>
        <button id="send">Wyślij</button>
      </div>
    </div>
  </div>

  <script>
    const chatContainer = document.getElementById('chat-container');
    const input = document.getElementById('input');
    const sendButton = document.getElementById('send');
    const recordButton = document.getElementById('record');
    const statusDiv = document.getElementById('status');
    const ws = new WebSocket(`ws://${window.location.host}/chat`);

    const activeResponses = {};
    let mediaRecorder;
    let isRecording = false;

    function createMessageElement(cssClass, label) {
      const messageDiv = document.createElement('div');
      messageDiv.className = `message ${cssClass}`;
      const labelSpan = document.createElement('span');
      labelSpan.className = 'model-label';
      labelSpan.textContent = label;
      const contentP = document.createElement('p');
      contentP.style.margin = '0';
      messageDiv.appendChild(labelSpan);
      messageDiv.appendChild(contentP);
      chatContainer.appendChild(messageDiv);
      chatContainer.scrollTop = chatContainer.scrollHeight;
      return contentP;
    }

    function handleSend() {
        const text = input.value.trim();
        if (!text) return;

        const userMessageContent = createMessageElement('user', 'Ty');
        userMessageContent.textContent = text;

        ws.send(JSON.stringify({ type: 'text', payload: text }));
        input.value = '';
        input.style.height = 'auto';

        activeResponses.azure = createMessageElement('model-response azure typing', 'Azure');
        activeResponses.gemini = createMessageElement('model-response gemini typing', 'Gemini');
    }

    async function toggleRecording() {
        if (isRecording) {
            mediaRecorder.stop();
            isRecording = false;
            recordButton.classList.remove('recording');
            recordButton.textContent = '●';
            statusDiv.textContent = 'Zakończono nagrywanie.';
        } else {
            try {
                const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
                mediaRecorder = new MediaRecorder(stream, { mimeType: 'audio/webm' });

                mediaRecorder.ondataavailable = (event) => {
                    if (event.data.size > 0 && ws.readyState === WebSocket.OPEN) {
                        ws.send(event.data);
                    }
                };

                mediaRecorder.onstart = () => {
                    isRecording = true;
                    recordButton.classList.add('recording');
                    recordButton.textContent = '■';
                    statusDiv.textContent = 'Nagrywanie...';
                };

                mediaRecorder.onstop = () => {
                    stream.getTracks().forEach(track => track.stop());
                };

                mediaRecorder.start(1000); // Wysyłaj dane co 1 sekundę
            } catch (err) {
                console.error("Błąd dostępu do mikrofonu:", err);
                statusDiv.textContent = `Błąd mikrofonu: ${err.message}`;
            }
        }
    }

    sendButton.onclick = handleSend;
    recordButton.onclick = toggleRecording;
    input.addEventListener('keydown', (e) => {
        if (e.key === 'Enter' && !e.shiftKey) {
            e.preventDefault();
            handleSend();
        }
    });

    ws.onmessage = (event) => {
      try {
        const data = JSON.parse(event.data);
        const { model, text, done, error } = data;

        if (!model || (!activeResponses[model] && !error)) return;
        const responseElement = activeResponses[model];

        if (text) responseElement.textContent += text;
        if (error) {
            const errorElement = activeResponses[model] || createMessageElement(`model-response ${model} error`, model);
            errorElement.textContent = error;
            errorElement.classList.add('error');
            if (errorElement.parentElement) errorElement.parentElement.classList.remove('typing');
            activeResponses[model] = null;
        }
        if (done) {
          if (responseElement && responseElement.parentElement) {
            responseElement.parentElement.classList.remove('typing');
          }
          activeResponses[model] = null;
        }
        chatContainer.scrollTop = chatContainer.scrollHeight;
      } catch (e) {
        // Ignoruj błędy parsowania, bo mogą to być wiadomości zwrotne serwera inne niż JSON
        console.warn("Could not parse WebSocket message as JSON:", event.data);
      }
    };

    ws.onopen = () => statusDiv.textContent = "Połączono.";
    ws.onclose = () => statusDiv.textContent = "Rozłączono.";
    ws.onerror = (error) => statusDiv.textContent = "Błąd WebSocket.";
  </script>
</body>
</html>